In [3]:
import numpy as np


class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    # -----------------------------
    # Gini Impurity
    # -----------------------------
    def gini(self, y):
        classes, counts = np.unique(y, return_counts=True)

        probabilities = counts / len(y)

        return 1 - np.sum(probabilities ** 2)

    # -----------------------------
    # Calculate Gini after a split
    # -----------------------------
    def split_gini(self, y_left, y_right):

        total = len(y_left) + len(y_right)

        left_weight = len(y_left) / total
        right_weight = len(y_right) / total

        return (
            left_weight * self.gini(y_left)
            + right_weight * self.gini(y_right)
        )

    # -----------------------------
    # Find the best split
    # -----------------------------
    def best_split(self, X, y):

        best_feature = None
        best_threshold = None
        best_gini = float("inf")

        n_samples, n_features = X.shape

        for feature in range(n_features):

            # Get possible values for this feature
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:

                left_indices = X[:, feature] <= threshold
                right_indices = X[:, feature] > threshold

                y_left = y[left_indices]
                y_right = y[right_indices]

                # Ignore invalid splits
                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                # Calculate Gini after split
                gini_value = self.split_gini(y_left, y_right)

                # Keep the best split
                if gini_value < best_gini:
                    best_gini = gini_value
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold

    # -----------------------------
    # Build the tree recursively
    # -----------------------------
    def build_tree(self, X, y, depth=0):

        # Find unique classes
        classes = np.unique(y)

        # Stopping conditions
        if (
            len(classes) == 1
            or depth >= self.max_depth
            or len(y) < self.min_samples_split
        ):
            # Return the most common class
            values, counts = np.unique(y, return_counts=True)
            prediction = values[np.argmax(counts)]

            return Node(value=prediction)

        # Find the best split
        feature, threshold = self.best_split(X, y)

        # If no useful split was found
        if feature is None:
            values, counts = np.unique(y, return_counts=True)
            prediction = values[np.argmax(counts)]

            return Node(value=prediction)

        # Divide the data
        left_indices = X[:, feature] <= threshold
        right_indices = X[:, feature] > threshold

        X_left = X[left_indices]
        y_left = y[left_indices]

        X_right = X[right_indices]
        y_right = y[right_indices]

        # Recursively build left and right branches
        left_child = self.build_tree(X_left, y_left, depth + 1)
        right_child = self.build_tree(X_right, y_right, depth + 1)

        # Return a decision node
        return Node(
            feature=feature,
            threshold=threshold,
            left=left_child,
            right=right_child
        )

    # -----------------------------
    # Train the tree
    # -----------------------------
    def fit(self, X, y):

        X = np.asarray(X)
        y = np.asarray(y)

        self.root = self.build_tree(X, y)

    # -----------------------------
    # Predict one observation
    # -----------------------------
    def predict_one(self, x, node):

        # If this is a leaf node
        if node.value is not None:
            return node.value

        # Follow the appropriate branch
        if x[node.feature] <= node.threshold:
            return self.predict_one(x, node.left)

        else:
            return self.predict_one(x, node.right)

    # -----------------------------
    # Predict multiple observations
    # -----------------------------
    def predict(self, X):

        X = np.asarray(X)

        return np.array([
            self.predict_one(x, self.root)
            for x in X
        ])

In [4]:
X = np.array([
    [2, 3],
    [3, 4],
    [4, 5],
    [7, 8],
    [8, 9],
    [9, 10]
])

y = np.array([
    0,
    0,
    0,
    1,
    1,
    1
])

tree = DecisionTree(max_depth=3)

tree.fit(X, y)

predictions = tree.predict(X)

print(predictions)

[0 0 0 1 1 1]
